<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "024"
iso_country = "Angola"
year = 2025
import_portfolio = "h6_ag6_by_sea_FM.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 122.170*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Angola': 

Number of trade records present in dataset: 37365
Number of partner countries present in dataset: 156
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$14130.0m
Total weight within the dataset: 11796.9 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,024,Angola,620,Portugal,Foreign Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,586950.67,566786.83,586950.67,kg,102337.720,102337.720,2.112377e+06,7771.0,4828.674041,1.019998e+07,['Container'],['Container'],32514.034504,32406.434019,32415.421578,32415.421578,32478.662285,32479.886657,34167.674445,63862.157989,37600.931857,34767.855112,34902.132348,35917.332239,33381.629838,61514.213157,39472.932000,36195.620744,35311.554247,37130.646583,32693.324464,68111.980937,48989.896544,43068.511762,39733.699287,38815.174163,32015.940375,71859.813585,57647.979754,53738.884838,42460.252302,41791.037982,31382.929500,73635.270009,70532.189677,70556.210857,45883.944031,44027.711182,30656.332388,72427.251159,69483.413734,74724.570728,56547.417243,44809.215725,30005.305142,63462.360939,66556.28527,71402.705042,53404.889010,45351.970311,29356.023820,57087.810714,58385.239497,69302.894271,56554.681877,45702.383367,3735
1,1,024,Angola,616,Poland,Foreign Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,69862.14,69410.00,69862.14,kg,78335.672,78335.672,2.514269e+05,11048.0,6864.906808,1.726022e+06,['Container'],['Container'],5501.965982,5483.758023,5485.278882,5485.278882,5495.980359,5496.187545,5781.791936,10806.638615,6362.761531,5883.353420,5906.075571,6077.865856,5648.778892,10409.323647,6679.538002,6124.957332,5975.357202,6283.180710,5532.305104,11525.785951,8289.981492,7287.975492,6723.664570,6568.233414,5417.679397,12159.987398,9755.086638,9093.596683,7185.046929,7071.803695,5310.562444,12460.426916,11935.329287,11939.394108,7764.397837,7450.289479,5187.608998,12256.008156,11757.857323,12644.756409,9568.851445,7582.534262,5077.443347,10738.985683,11262.53395,12082.636319,9037.078514,7674.378209,4967.573139,9660.295849,9879.844396,11727.310147,9570.080753,7733.674471,3735


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $2527.2m (17.9%), ... 

                              cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                              
10 | Cereals               859090933.5  2.674377e+09      6.1     22.7
02 | Meat and edible meat  451022873.0  2.188102e+08      3.2      1.9
15 | Animal, vegetable or  355439953.3  3.536910e+08      2.5      3.0
17 | Sugars and sugar con  172285954.6  3.621358e+08      1.2      3.1
04 | Dairy produce; birds  106986649.0  3.908512e+07      0.8      0.3
22 | Beverages, spirits a   97275363.4  1.031653e+08      0.7      0.9
21 | Miscellaneous edible   91211195.5  2.690831e+07      0.6      0.2
11 | Products of the mill   76120684.0  1.402691e+08      0.5      1.2
19 | Preparations of cere   72993660.0  2.364423e+07      0.5      0.2
16 | Meat, fish, crustace   58793870.1  1.309702e+07      0.4      0.1
07 | Vegetables and certa   40033616.5  6.2873

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $2024.9m (14.3%), ... 

                      cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                     
Agriculture       1.431561e+09  1.784551e+09     10.1     15.1
Food & Beverages  5.808982e+08  6.114175e+08      4.1      5.2
Fishing           1.245463e+07  1.652503e+06      0.1      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $1862.6m (13.2%), ... 

                                                 cifvalue   vol_kg_vtw  \
gtap_desc                                                                
OMT - Meat products n.e.c.                    433067825.2  209722406.9   
VOL - Vegetable oils and fats                 357860014.8  369043949.5   
OFD - Food products n.e.c.                    313956843.4  206011770.5   
PCR - Processed rice                          226148455.8  392819867.7   
SGR - Sugar                                   142134933.3  311938033.0   
GRO - Cereal grains n.e.c.                    121732860.8  503538634.6   
MIL - Dairy products                          104717920.0   37160007.9   
CMT - Bovine meat prods                        55647031.6   16578856.2   
PDR - Paddy rice                               40725135.3   70739560.0   
V_F - Vegetables, fruit, nuts                  39144900.7   61214960.8   
OCR - Crops n.e.c.                           

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 17.9% 
GTAP 13.2% 
EORA 14.3%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46329/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,024,Angola,620,Portugal,Foreign Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,586950.67,566786.83,586950.67,kg,102337.720,102337.720,2.112377e+06,7771.0,4828.674041,1.019998e+07,['Container'],['Container'],32514.034504,32406.434019,32415.421578,32415.421578,32478.662285,32479.886657,34167.674445,63862.157989,37600.931857,34767.855112,34902.132348,35917.332239,33381.629838,61514.213157,39472.932000,36195.620744,35311.554247,37130.646583,32693.324464,68111.980937,48989.896544,43068.511762,39733.699287,38815.174163,32015.940375,71859.813585,57647.979754,53738.884838,42460.252302,41791.037982,31382.929500,73635.270009,70532.189677,70556.210857,45883.944031,44027.711182,30656.332388,72427.251159,69483.413734,74724.570728,56547.417243,44809.215725,30005.305142,63462.360939,66556.28527,71402.705042,53404.889010,45351.970311,29356.023820,57087.810714,58385.239497,69302.894271,56554.681877,45702.383367,3735,NaN
1,1,024,Angola,616,Poland,Foreign Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,69862.14,69410.00,69862.14,kg,78335.672,78335.672,2.514269e+05,11048.0,6864.906808,1.726022e+06,['Container'],['Container'],5501.965982,5483.758023,5485.278882,5485.278882,5495.980359,5496.187545,5781.791936,10806.638615,6362.761531,5883.353420,5906.075571,6077.865856,5648.778892,10409.323647,6679.538002,6124.957332,5975.357202,6283.180710,5532.305104,11525.785951,8289.981492,7287.975492,6723.664570,6568.233414,5417.679397,12159.987398,9755.086638,9093.596683,7185.046929,7071.803695,5310.562444,12460.426916,11935.329287,11939.394108,7764.397837,7450.289479,5187.608998,12256.008156,11757.857323,12644.756409,9568.851445,7582.534262,5077.443347,10738.985683,11262.53395,12082.636319,9037.078514,7674.378209,4967.573139,9660.295849,9879.844396,11727.310147,9570.080753,7733.674471,3735,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $2071.9m of $14130.0m (14.7%)
Volume: 3760.0 of 11796.9 million kilograms (31.9%)
Value-to-Weight: 0.55 from 1.2 US$ per kg (+-54.0%)
Average distance to market (volume-weighted): 5653.0 from 6766.5 nm (+-16.5%)

            food_group  N_com     cifvalue    vol_kg_vtw  pct_val  pct_vol
0  Unprocessed Cereals     54  592217342.5  2.210817e+09      4.2     18.7
1    Processed Cereals    613  508793560.8  6.478084e+08      3.6      5.5
2        Meat and Prep    182  389180589.8  2.009942e+08      2.8      1.7
3        Fats and Oils    110  250119314.6  2.289123e+08      1.8      1.9
4      Sugar and Honey    120  172396491.5  3.621675e+08      1.2      3.1
5                Dairy    167  105088449.1  3.842696e+07      0.7      0.3
6           Vegetables    325   51642064.9  6.965585e+07      0.4      0.6
7               Fruits    175    2484576.4  1.168713e+06      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,7.786043e+07,1.444817e+08,8.547682e+07,7.919777e+07,7.949564e+07,8.177662e+07,7.161802e+07,1.669016e+08,1.596783e+08,1.599779e+08,1.040871e+08,1.000750e+08,6.705165e+07,1.295935e+08,1.325507e+08,1.569998e+08,1.280888e+08,1.039472e+08
1,Fats and Oils,4.440635e+08,7.151463e+08,4.726739e+08,4.437148e+08,4.449492e+08,4.605014e+08,4.177307e+08,8.075037e+08,7.799343e+08,8.061080e+08,5.651147e+08,5.364910e+08,3.992160e+08,6.698038e+08,6.853821e+08,7.933200e+08,6.617543e+08,5.524231e+08
2,Fruits,2.390242e+06,4.438968e+06,2.625775e+06,2.432032e+06,2.441192e+06,2.511169e+06,2.197829e+06,5.127351e+06,4.904503e+06,4.912381e+06,3.201369e+06,3.073211e+06,2.057713e+06,3.984564e+06,4.073428e+06,4.824520e+06,3.932791e+06,3.190292e+06
3,Meat and Prep,3.887502e+08,7.266057e+08,4.278129e+08,3.955789e+08,3.971067e+08,4.086573e+08,3.570662e+08,8.378015e+08,8.024955e+08,8.027688e+08,5.220547e+08,5.009350e+08,3.340046e+08,6.495291e+08,6.642909e+08,7.885089e+08,6.434633e+08,5.199889e+08
4,Processed Cereals,1.632535e+09,3.021338e+09,1.791696e+09,1.659229e+09,1.665550e+09,1.714408e+09,1.502103e+09,3.491625e+09,3.343813e+09,3.350821e+09,2.189700e+09,2.093895e+09,1.407280e+09,2.714677e+09,2.774933e+09,3.286204e+09,2.686849e+09,2.172175e+09
5,Sugar and Honey,4.767105e+08,8.604024e+08,5.196439e+08,4.829707e+08,4.847497e+08,4.993584e+08,4.405351e+08,9.997933e+08,9.574070e+08,9.635371e+08,6.379107e+08,6.041381e+08,4.143417e+08,7.833157e+08,7.995953e+08,9.414718e+08,7.740013e+08,6.255991e+08
6,Unprocessed Cereals,3.982281e+09,7.387181e+09,4.373340e+09,4.048340e+09,4.063818e+09,4.182977e+09,3.662618e+09,8.531718e+09,8.171828e+09,8.185184e+09,5.343792e+09,5.112867e+09,3.430204e+09,6.629553e+09,6.777419e+09,8.030796e+09,6.564118e+09,5.304532e+09
7,Vegetables,1.415786e+08,2.630459e+08,1.555412e+08,1.440100e+08,1.445597e+08,1.487392e+08,1.301788e+08,3.038919e+08,2.907971e+08,2.913209e+08,1.897267e+08,1.820318e+08,1.218692e+08,2.358095e+08,2.411407e+08,2.857629e+08,2.331951e+08,1.889516e+08


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.63,0.07,0.01,0.02,0.04,0.91,0.84,0.84,0.31,0.27,0.60,0.62,0.86,0.58,0.35
1,Fats and Oils,1.08,0.11,-0.00,0.00,0.07,1.56,1.45,1.55,0.59,0.47,1.08,1.14,1.58,1.05,0.61
2,Fruits,0.82,0.09,0.02,0.02,0.05,1.18,1.09,1.09,0.40,0.35,0.78,0.81,1.11,0.75,0.46
3,Meat and Prep,0.87,0.10,0.02,0.02,0.05,1.24,1.14,1.15,0.42,0.37,0.81,0.85,1.17,0.80,0.48
4,Processed Cereals,2.73,0.31,0.05,0.06,0.16,3.91,3.62,3.63,1.35,1.16,2.57,2.69,3.69,2.51,1.50
5,Sugar and Honey,2.23,0.25,0.04,0.05,0.13,3.24,3.00,3.03,1.14,0.95,2.14,2.23,3.06,2.09,1.23
6,Unprocessed Cereals,5.75,0.66,0.11,0.14,0.34,8.22,7.61,7.64,2.84,2.45,5.40,5.65,7.77,5.29,3.16
7,Vegetables,2.35,0.27,0.05,0.06,0.14,3.36,3.11,3.12,1.15,1.00,2.21,2.31,3.17,2.16,1.30
